# MOSAIC Data Warehouse Naming Convention Detector
**Detection-only.** Read-only against metadata only — no table data is read. Every finding requires steward review before any action.

## What makes this detector different

All other MOSAIC DQ detectors validate **data content** (dates, emails, codes). This detector validates **names** — table names, column names, and schema names — against the MOSAIC DW naming convention procedure. Most checks run entirely against `information_schema` with no data reads, making this detector extremely fast at scale.

## Coverage by procedure section

| Section | What it checks |
|---|---|
| §1 Universal | snake_case compliance; descriptive names; reserved words; environment in names; quoted mixed-case |
| §2a Keys | Foreign key encoding table name; source key naming; lineage column presence |
| §2b Dates/times | `_dt`/`_ts` suffixes; missing `_at_est` canonical; wrong date column types |
| §2c Booleans | `is_`/`has_`/`was_` prefix compliance; semantic correctness |
| §2d Measures | `_amount` suffix; code+description pairing; Bronze source prefixes in Silver+ |
| §2e Audit | Missing `ingested_at`/`source_file`/`batch_id` on Bronze; leading underscore misuse |
| §2f Sensitive | Opaque column names (`col_17`, `x_1`) |
| §3 Table/view | `dim_`/`fact_` layer and form correctness; `_vw`/`_view` suffix prohibition; abbreviated prefixes |

## AI use

AI is used for two checks that are too ambiguous for deterministic regex:
1. **NON_DESCRIPTIVE_NAME** — distinguishes a genuine abbreviation (`npi`, `icd`) from an opaque one (`mcl`, `tbl1`) by checking against known healthcare domain terms
2. **TABLE_ENCODED_IN_KEY** — confirms whether a foreign key column name genuinely encodes a table name vs. a legitimate composite term

| Cell | What it does |
|---|---|
| 1-2 | Overview and Rule Reference |
| 3 | Config / Widgets |
| 4 | Constants: naming rules, reserved words, approved patterns, standardization rules |
| 5 | Discovery: read information_schema; build table+column inventory |
| 6 | Schema Analyzer: apply all deterministic naming checks (no data reads) |
| 7 | AI Classifier: validate ambiguous names; detect opaque abbreviations |
| 8 | Rule Engine: consolidate findings; apply severity; deduplicate |
| 9 | Write Results |
| 10 | Summary: by section → by table → blocking → work queue |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed. No rows are read from source tables except for minimal type-checking on boolean and audit columns.


# MOSAIC Naming Convention Detector — Rule Reference

---

## §1 Universal identifier rules
| Rule tag | What it detects | Required action |
|---|---|---|
| `NON_SNAKE_CASE` | camelCase, PascalCase, kebab-case, spaces, or UPPER_CASE in table/column name | Rename to snake_case in new models; steward approval required for certified columns (§1.7) |
| `NON_DESCRIPTIVE_NAME` | Single-char names, tbl/col prefixes, numbered suffixes (tbl1), or opaque abbreviations not in glossary | Rename to descriptive name; expand abbreviation or register in glossary |
| `RESERVED_WORD` | Column/table name is a SQL reserved word (date, order, select, group, etc.) | Suffix or qualify (service_date, order_status) — bare reserved words prohibited |
| `ENV_IN_NAME` | Environment token (prod, dev, staging, uat, test) embedded in table/column name | Remove; express environment via catalog/workspace/bundle target (§1.5) |
| `QUOTED_MIXED_CASE` | UC stores name in mixed-case — was created with quoted identifier | Recreate with unquoted lowercase; quoted mixed-case prohibited in new models (§1.8) |

## §2a Keys and relationships
| Rule tag | What it detects | Required action |
|---|---|---|
| `TABLE_ENCODED_IN_KEY` | Foreign key column name encodes the referenced table (claims_table_member_id) | Rename to referenced key name (member_id) — do not encode table name in keys (§2a.2) |
| `AUDIT_COL_MISSING` | Bronze table without ingested_at, source_file, or batch_id | Add mandatory audit columns (§2e.1); audit columns required on all Bronze tables |

## §2b Dates and times
| Rule tag | What it detects | Required action |
|---|---|---|
| `DT_TS_SUFFIX` | Column using _dt or _ts suffix instead of _date, _at_est, or _at_utc | Rename to standard suffix (§2b.1); never use _dt or _ts |
| `MISSING_EST_COMPANION` | Timestamp column without _at_est canonical companion | Add _at_est Eastern canonical column (§2b.2) |
| `DATE_WRONG_TYPE` | Column named _date but stored as STRING or TIMESTAMP instead of DATE | Cast to DATE type in Silver ETL |

## §2c Booleans
| Rule tag | What it detects | Required action |
|---|---|---|
| `BOOL_WRONG_PREFIX` | Boolean/indicator column not prefixed with is_, has_, or was_ | Rename to is_{condition}, has_{attribute}, or was_{event} (§2c) |
| `BOOL_WRONG_TYPE` | Column named is_*/has_*/was_* but stored as STRING instead of BOOLEAN/INT | Cast to BOOLEAN or 0/1 INT in Silver ETL (§2c.1) |

## §2d Measures, amounts, and codes
| Rule tag | What it detects | Required action |
|---|---|---|
| `AMOUNT_NO_SUFFIX` | Numeric column with monetary naming but no _amount suffix | Add _amount suffix (paid_amount, allowed_amount) (§2d.1) |
| `CODE_WITHOUT_DESCRIPTION` | _code column present but no matching _description column in same table | Add {domain}_description column (§2d.2) |
| `DESCRIPTION_WITHOUT_CODE` | _description column present but no matching _code column in same table | Add {domain}_code column or rename (§2d.2) |
| `SOURCE_PREFIX_IN_SILVER` | source_*/epic_*/athena_* prefix column in Silver or Gold layer | Rename to canonical domain name; source prefixes Bronze-only (§2d.3) |

## §2e Audit columns
| Rule tag | What it detects | Required action |
|---|---|---|
| `LEADING_UNDERSCORE_MISUSE` | Column with leading underscore not in approved audit metadata set | Rename; leading underscore reserved for technical metadata (§2e.1) |

## §2f Sensitive columns
| Rule tag | What it detects | Required action |
|---|---|---|
| `OPAQUE_COLUMN_NAME` | Opaque name pattern: col_N, field_N, x_N, val_N | Rename to governed descriptive name (§2f.1); use governed names with UC phi_class tags |

## §3 Table and view naming
| Rule tag | What it detects | Required action |
|---|---|---|
| `DIM_FACT_WRONG_LAYER` | dim_/fact_ prefix on Bronze or Silver table | Remove prefix; dim_/fact_ for Gold star-schema only (§3a.2) |
| `DIM_FACT_SUFFIX_FORM` | provider_dim, claims_fact (suffix form instead of prefix) | Rename to dim_provider, fact_claims (§3a.5) |
| `VIEW_SUFFIX_PRESENT` | _vw, _view, or vw_ in table/view name | Rename; view-indicating suffixes prohibited (§3a.4) |
| `DIM_FACT_ABBREVIATED` | fct_, d_, f_, dmn_ prefix instead of full dim_/fact_ | Rename to full dim_/fact_ prefix (§3a.1) |
| `WRONG_PREFIX_NONDIM_GOLD` | dim_/fact_ on Gold object not designed as dimensional | Remove prefix or confirm dimensional design in data dictionary (§3a.2) |
| `STG_PREFIX_WRONG_LAYER` | stg_ prefix on Bronze or Silver (pipeline staging) used on Gold table name | Clarify: stg_ on Gold (data_marts) is analytical staging, not pipeline staging (§3a.7) |

---

## Enforcement
- Renames of certified Gold columns require steward approval and consumer impact assessment (§1.7).
- Quoted mixed-case identifiers are prohibited in new models (§1.8).
- `dim_`/`fact_` prefix violations on Gold objects are certification blockers.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set, Optional
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "naming_convention_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "false").strip().lower() == "true",

    # Layer of the tables being scanned -- drives which rules apply
    # (e.g. dim_/fact_ allowed on Gold only; audit cols required on Bronze)
    "layer": _w("layer", "Unknown"),   # Bronze | Silver | Gold | Unknown

    # Whether to check table names in addition to column names
    "check_table_names": _w("check_table_names", "true").strip().lower() == "true",

    # Whether to flag single-schema scope (for table naming context like dim_/fact_ layer check)
    # When Unknown, use heuristics from table names themselves
    "schema_is_gold":   _w("schema_is_gold",   "false").strip().lower() == "true",
    "schema_is_bronze": _w("schema_is_bronze", "false").strip().lower() == "true",
    "schema_is_silver": _w("schema_is_silver", "false").strip().lower() == "true",

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "naming_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

def _detect_layer(layer_cfg: str, schema: str, tbl: str) -> str:
    if layer_cfg != "Unknown":
        return layer_cfg
    # Heuristic: infer from catalog/schema/table name tokens
    s = (schema + " " + tbl).lower()
    if any(t in s for t in ("bronze", "_raw", "_landing", "_ingest")):
        return "Bronze"
    if any(t in s for t in ("silver", "_conform", "_curated")):
        return "Silver"
    if any(t in s for t in ("gold", "data_marts", "_mart", "_dim", "_fact")):
        return "Gold"
    return "Unknown"

print(f"Run     : {RUN_ID}")
print(f"Scope   : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer   : {CFG['layer']}")
print(f"AI      : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off'}")
print(f"Results : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# CONSTANTS — naming rules, patterns, reserved words, approved sets
# ---------------------------------------------------------------------------

TAGS = {
    # §1
    "NON_SNAKE_CASE":           "§1-Universal",
    "NON_DESCRIPTIVE_NAME":     "§1-Universal",
    "RESERVED_WORD":            "§1-Universal",
    "ENV_IN_NAME":              "§1-Universal",
    "QUOTED_MIXED_CASE":        "§1-Universal",
    # §2a
    "TABLE_ENCODED_IN_KEY":     "§2a-Keys",
    "AUDIT_COL_MISSING":        "§2e-Audit",
    # §2b
    "DT_TS_SUFFIX":             "§2b-Dates",
    "MISSING_EST_COMPANION":    "§2b-Dates",
    "DATE_WRONG_TYPE":          "§2b-Dates",
    # §2c
    "BOOL_WRONG_PREFIX":        "§2c-Booleans",
    "BOOL_WRONG_TYPE":          "§2c-Booleans",
    # §2d
    "AMOUNT_NO_SUFFIX":         "§2d-Measures",
    "CODE_WITHOUT_DESCRIPTION": "§2d-Measures",
    "DESCRIPTION_WITHOUT_CODE": "§2d-Measures",
    "SOURCE_PREFIX_IN_SILVER":  "§2d-Measures",
    # §2e
    "LEADING_UNDERSCORE_MISUSE":"§2e-Audit",
    # §2f
    "OPAQUE_COLUMN_NAME":       "§2f-Sensitive",
    # §3
    "DIM_FACT_WRONG_LAYER":     "§3-Table-naming",
    "DIM_FACT_SUFFIX_FORM":     "§3-Table-naming",
    "VIEW_SUFFIX_PRESENT":      "§3-Table-naming",
    "DIM_FACT_ABBREVIATED":     "§3-Table-naming",
    "STG_PREFIX_WRONG_LAYER":   "§3-Table-naming",
}

SEVERITY: Dict[str, int] = {
    "RESERVED_WORD":            10,
    "NON_SNAKE_CASE":            9,
    "QUOTED_MIXED_CASE":         9,
    "DIM_FACT_WRONG_LAYER":      9,
    "VIEW_SUFFIX_PRESENT":       8,
    "AUDIT_COL_MISSING":         8,
    "DT_TS_SUFFIX":              8,
    "BOOL_WRONG_TYPE":           7,
    "DATE_WRONG_TYPE":           7,
    "AMOUNT_NO_SUFFIX":          7,
    "SOURCE_PREFIX_IN_SILVER":   7,
    "DIM_FACT_SUFFIX_FORM":      7,
    "DIM_FACT_ABBREVIATED":      7,
    "MISSING_EST_COMPANION":     6,
    "CODE_WITHOUT_DESCRIPTION":  6,
    "DESCRIPTION_WITHOUT_CODE":  6,
    "OPAQUE_COLUMN_NAME":        6,
    "TABLE_ENCODED_IN_KEY":      6,
    "BOOL_WRONG_PREFIX":         5,
    "ENV_IN_NAME":               5,
    "LEADING_UNDERSCORE_MISUSE": 5,
    "NON_DESCRIPTIVE_NAME":      4,
    "STG_PREFIX_WRONG_LAYER":    3,
}

# ── §1 Universal naming patterns ──────────────────────────────────────────────
# snake_case: only lowercase letters, digits, underscores; must start with letter/underscore
RE_SNAKE_CASE   = re.compile(r"^[a-z_][a-z0-9_]*$")
RE_CAMEL_CASE   = re.compile(r"[a-z][A-Z]")           # camelCase signal
RE_PASCAL_CASE  = re.compile(r"^[A-Z]")               # PascalCase signal
RE_UPPER_CASE   = re.compile(r"^[A-Z_]+$")            # ALL_CAPS
RE_KEBAB_CASE   = re.compile(r"-")                    # kebab-case
RE_SPACE        = re.compile(r" ")                    # space in name
RE_MIXED_CASE   = re.compile(r"[A-Z]")               # any uppercase = was quoted

# Environment tokens (§1.5)
ENV_TOKENS = {"prod", "production", "dev", "development", "staging", "stage",
              "uat", "qa", "test", "sandbox", "local", "sit", "preprod"}

# Single-char or trivially non-descriptive
RE_OPAQUE_NUMBERED = re.compile(r"^(col|field|x|val|tbl|t|f|d|tmp|temp)_?[0-9]+$", re.I)
RE_SINGLE_CHAR     = re.compile(r"^[a-z]$")

# ── SQL reserved words (subset — most commonly misused as column names) ───────
SQL_RESERVED = {
    "select","from","where","group","order","having","join","on","and","or","not",
    "in","is","null","true","false","case","when","then","else","end","as","by",
    "asc","desc","limit","offset","union","all","distinct","insert","update",
    "delete","drop","create","alter","table","view","index","schema","database",
    "column","row","key","primary","foreign","default","check","unique","constraint",
    "transaction","commit","rollback","grant","revoke","user","role","set",
    # Business-word reserved words commonly used as bare names
    "date","time","timestamp","year","month","day","hour","minute","second",
    "name","type","status","level","value","code","id","key","flag","count",
    "sum","avg","min","max","rank","row","first","last","next","current",
    "mode","data","size","length","class","group","order","number","amount",
}

# ── §2b Date/time patterns ────────────────────────────────────────────────────
RE_DT_SUFFIX   = re.compile(r"_(dt|ts)$", re.I)
RE_DATE_SUFFIX = re.compile(r"_date$", re.I)
RE_AT_EST      = re.compile(r"_at_est$", re.I)
RE_AT_UTC      = re.compile(r"_at_utc$", re.I)
RE_AT_SUFFIX   = re.compile(r"_at_[a-z]{2,4}$", re.I)  # _at_est, _at_utc, _at_ct
RE_TIMESTAMP_COL = re.compile(
    r"_(at|time|datetime|timestamp|created|updated|modified|ingested|processed)$", re.I
)

# ── §2c Boolean patterns ──────────────────────────────────────────────────────
RE_BOOL_PREFIX = re.compile(r"^(is|has|was)_", re.I)
RE_BOOL_LIKE   = re.compile(
    r"_(flag|indicator|ind|active|eligible|enrolled|deceased|deleted|"
    r"valid|enabled|disabled|allowed|denied|approved|rejected|archived)$",
    re.I
)
BOOL_TYPES     = {"BOOLEAN", "INT", "TINYINT", "SMALLINT", "BIGINT"}

# ── §2d Measures patterns ─────────────────────────────────────────────────────
RE_AMOUNT_LIKE = re.compile(
    r"_(paid|allowed|billed|charged|cost|payment|premium|copay|"
    r"deductible|coinsurance|reimbursed|revenue|fee|price|rate|amount)$",
    re.I
)
RE_AMOUNT_SUFFIX = re.compile(r"_amount$", re.I)
RE_CODE_SUFFIX   = re.compile(r"_code$", re.I)
RE_DESC_SUFFIX   = re.compile(r"_(description|desc)$", re.I)
NUMERIC_TYPES    = {"INT","BIGINT","FLOAT","DOUBLE","DECIMAL","NUMERIC","SMALLINT","TINYINT"}

# Source prefixes that should only appear in Bronze (§2d.3)
RE_SOURCE_PREFIX = re.compile(
    r"^(source_|epic_|athena_|caremore_|mpg_|cerner_|allscripts_|"
    r"nextgen_|meditech_|eclinicalworks_|practicefusion_|emr_|ehr_)",
    re.I
)

# ── §2e Audit columns ─────────────────────────────────────────────────────────
REQUIRED_AUDIT_BRONZE = {"ingested_at", "source_file", "batch_id"}
APPROVED_LEADING_UNDERSCORE = {
    "_ingested_at", "_source_file", "_batch_id", "_updated_at",
    "_record_hash", "_dlt_id", "_commit_version", "_file_path",
    "_rescued_data",   # Delta RESCUED_DATA column
}

# ── §3 Table naming patterns ──────────────────────────────────────────────────
RE_DIM_PREFIX     = re.compile(r"^dim_", re.I)
RE_FACT_PREFIX    = re.compile(r"^fact_", re.I)
RE_STG_PREFIX     = re.compile(r"^stg_", re.I)
RE_DIM_SUFFIX     = re.compile(r"_dim$", re.I)
RE_FACT_SUFFIX    = re.compile(r"_fact$", re.I)
RE_VIEW_SUFFIX    = re.compile(r"(_vw|_view)$", re.I)
RE_VIEW_PREFIX    = re.compile(r"^vw_", re.I)
RE_DIM_ABBREV     = re.compile(r"^(d_|dmn_|dim(?!ension)(?!_))", re.I)
RE_FACT_ABBREV    = re.compile(r"^(f_|fct_)", re.I)

STANDARDIZATION_RULES: Dict[str, list] = {

    "NON_SNAKE_CASE": [
        {"option_key": "rename_to_snake_case",
         "label":      "Rename to snake_case in new models",
         "sql":        "-- ALTER TABLE ... RENAME COLUMN <col> TO <col_snake>",
         "notes":      "All identifiers must be snake_case (§1.1). "
                       "Renames of certified Gold columns require steward approval and "
                       "consumer impact assessment (§1.7). "
                       "SQL and dbt must use lowercase unquoted identifiers (§1.8)."},
    ],

    "NON_DESCRIPTIVE_NAME": [
        {"option_key": "rename_descriptive",
         "label":      "Rename to descriptive name or register abbreviation in glossary",
         "sql":        "-- Rename column to descriptive name per data dictionary.",
         "notes":      "Names must be descriptive and unambiguous (§1.2). "
                       "Abbreviations must appear in the approved glossary or be expanded (§1.3)."},
    ],

    "RESERVED_WORD": [
        {"option_key": "suffix_reserved_word",
         "label":      "Add domain suffix to disambiguate reserved word",
         "sql":        "-- Rename: date → service_date, order → order_status, group → group_id",
         "notes":      "Reserved words prohibited as bare identifiers (§1.4). "
                       "Suffix or qualify with domain context."},
    ],

    "ENV_IN_NAME": [
        {"option_key": "remove_env_token",
         "label":      "Remove environment token from name",
         "sql":        "-- Rename: claims_prod → claims, member_dev_v2 → member",
         "notes":      "Environment expressed via catalog/workspace/bundle target, "
                       "not embedded in production table or column names (§1.5)."},
    ],

    "QUOTED_MIXED_CASE": [
        {"option_key": "recreate_unquoted",
         "label":      "Recreate object with unquoted lowercase identifier",
         "sql":        "-- CREATE OR REPLACE TABLE <lowercase_name> AS SELECT * FROM <quoted_name>",
         "notes":      "Quoted mixed-case identifiers prohibited in new models (§1.8). "
                       "UC stores names consistently when using lowercase unquoted identifiers."},
    ],

    "TABLE_ENCODED_IN_KEY": [
        {"option_key": "rename_to_referenced_key",
         "label":      "Rename to referenced key name without table prefix",
         "sql":        "-- claims_table_member_id → member_id (§2a.2)",
         "notes":      "Foreign keys use the same name as the referenced key (§2a.2). "
                       "Do not encode table names in key columns."},
    ],

    "AUDIT_COL_MISSING": [
        {"option_key": "add_audit_columns",
         "label":      "Add mandatory audit columns to Bronze table",
         "sql":        "-- ADD COLUMN ingested_at TIMESTAMP, source_file STRING, batch_id STRING",
         "notes":      "Audit columns mandatory on Bronze (§2e.1): "
                       "ingested_at, source_file, batch_id."},
    ],

    "DT_TS_SUFFIX": [
        {"option_key": "rename_to_standard_suffix",
         "label":      "Rename to _date, _at_est, or _at_utc suffix",
         "sql":        "-- event_dt → event_date  |  created_ts → created_at_est",
         "notes":      "Never use _dt or _ts suffixes (§2b.1). "
                       "Use _date for DATE columns, _at_est for canonical Eastern timestamps, "
                       "_at_utc for UTC companion."},
    ],

    "MISSING_EST_COMPANION": [
        {"option_key": "add_est_canonical",
         "label":      "Add _at_est Eastern canonical companion column",
         "sql":        "-- SILVER: CONVERT_TIMEZONE('US/Eastern', col) AS <event>_at_est",
         "notes":      "Business event timestamps must have _at_est canonical (§2b.2). "
                       "When source is non-Eastern, retain original in _at_utc or source-tz companion."},
    ],

    "DATE_WRONG_TYPE": [
        {"option_key": "cast_to_date",
         "label":      "Cast _date column to DATE type in Silver ETL",
         "sql":        "-- SILVER: CAST(col AS DATE) AS <col>_date",
         "notes":      "DATE columns must use typed DATE (§2b.3). "
                       "Store as ISO 8601 YYYY-MM-DD."},
    ],

    "BOOL_WRONG_PREFIX": [
        {"option_key": "rename_with_bool_prefix",
         "label":      "Rename boolean column with is_/has_/was_ prefix",
         "sql":        "-- active_flag → is_active  |  hospice_ind → has_hospice_election",
         "notes":      "Boolean columns must use is_{condition}, has_{attribute}, or was_{event} (§2c)."},
    ],

    "BOOL_WRONG_TYPE": [
        {"option_key": "cast_to_bool_type",
         "label":      "Cast is_/has_/was_ column to BOOLEAN or INT (0/1) in Silver ETL",
         "sql":        "-- SILVER: CAST(col AS BOOLEAN) AS col  -- or 0/1 INT",
         "notes":      "Boolean columns must be BOOLEAN or 0/1 INT, not STRING (§2c.1). "
                       "Silver = source of truth; Gold keeps 0/1 by default."},
    ],

    "AMOUNT_NO_SUFFIX": [
        {"option_key": "add_amount_suffix",
         "label":      "Add _amount suffix to monetary column",
         "sql":        "-- paid → paid_amount  |  allowed → allowed_amount",
         "notes":      "Currency columns must include _amount in the name (§2d.1). "
                       "Store USD values as numeric type."},
    ],

    "CODE_WITHOUT_DESCRIPTION": [
        {"option_key": "add_description_column",
         "label":      "Add paired {domain}_description column",
         "sql":        "-- ADD COLUMN {domain}_description STRING",
         "notes":      "Codes and descriptions are paired consistently (§2d.2). "
                       "{domain}_code always accompanied by {domain}_description."},
    ],

    "DESCRIPTION_WITHOUT_CODE": [
        {"option_key": "add_code_column",
         "label":      "Add paired {domain}_code column",
         "sql":        "-- ADD COLUMN {domain}_code STRING",
         "notes":      "Codes and descriptions are paired consistently (§2d.2). "
                       "{domain}_description always accompanied by {domain}_code."},
    ],

    "SOURCE_PREFIX_IN_SILVER": [
        {"option_key": "rename_to_canonical",
         "label":      "Rename to canonical domain name in Silver",
         "sql":        "-- epic_patient_id → patient_id  |  source_member_key → member_id",
         "notes":      "Bronze may retain source_* prefixes; Silver+ converges to "
                       "canonical names per domain dictionary (§2d.3)."},
    ],

    "LEADING_UNDERSCORE_MISUSE": [
        {"option_key": "rename_remove_underscore",
         "label":      "Rename to remove leading underscore (not an approved audit column)",
         "sql":        "-- _status → status  |  _type → record_type",
         "notes":      "Leading underscore denotes technical metadata — not business semantics (§2e.1). "
                       "Only approved: _ingested_at, _source_file, _batch_id, _updated_at, _record_hash."},
    ],

    "OPAQUE_COLUMN_NAME": [
        {"option_key": "rename_descriptive",
         "label":      "Rename to governed descriptive name with UC phi_class tag if sensitive",
         "sql":        "-- col_17 → ssn_token  |  x_1 → member_dob",
         "notes":      "Column names must not obscure sensitivity (§2f.1). "
                       "Use governed names; apply UC phi_class tags for PHI columns."},
    ],

    "DIM_FACT_WRONG_LAYER": [
        {"option_key": "remove_prefix",
         "label":      "Remove dim_/fact_ prefix from non-Gold table",
         "sql":        "-- Rename to domain name without dim_/fact_ prefix.",
         "notes":      "dim_ and fact_ prefixes for Gold star-schema only (§3a.2). "
                       "Do not apply to Bronze, Silver, or non-dimensional Gold products."},
    ],

    "DIM_FACT_SUFFIX_FORM": [
        {"option_key": "convert_to_prefix_form",
         "label":      "Convert suffix form to prefix form",
         "sql":        "-- provider_dim → dim_provider  |  claims_fact → fact_claims",
         "notes":      "Use prefix form: dim_{entity}, fact_{entity} (§3a.5). "
                       "Never use suffix forms: {entity}_dim, {entity}_fact."},
    ],

    "VIEW_SUFFIX_PRESENT": [
        {"option_key": "remove_view_suffix",
         "label":      "Remove _vw/_view/vw_ from object name",
         "sql":        "-- member_summary_vw → member_summary  |  vw_claims → claims",
         "notes":      "View-indicating suffixes and prefixes prohibited (§3a.4). "
                       "Layer, schema, and UC object type convey consumption shape. "
                       "Legacy views require steward-tracked sunset."},
    ],

    "DIM_FACT_ABBREVIATED": [
        {"option_key": "expand_to_full_prefix",
         "label":      "Replace abbreviated prefix with full dim_/fact_",
         "sql":        "-- fct_claims → fact_claims  |  d_provider → dim_provider",
         "notes":      "Full words required: dim_ and fact_ (§3a.1). "
                       "Never abbreviate to fct_, d_, f_, dmn_."},
    ],

    "STG_PREFIX_WRONG_LAYER": [
        {"option_key": "clarify_stg_context",
         "label":      "Clarify stg_ context: pipeline staging vs. analytical Gold staging",
         "sql":        "-- stg_ on Gold data_marts = analytical staging (correct per §3a.7). "
                       "-- stg_ on Bronze/Silver = pipeline staging (use medallion naming instead).",
         "notes":      "stg_ on Gold (data_marts) is analytical staging for report builds (§3a.7). "
                       "Document grain, upstream refs, and retirement criteria in data dictionary."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} rules")
print(f"SQL reserved words: {len(SQL_RESERVED)}")
print(f"Required Bronze audit cols: {sorted(REQUIRED_AUDIT_BRONZE)}")


In [0]:
# ---------------------------------------------------------------------------
# DISCOVERY -- reads information_schema ONLY. No data reads.
# Builds: tables_meta (table name + type + layer)
#         cols_meta (col name + data_type per table)
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

# Include views unless skip_views=True
view_filter = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""

tables_meta: Dict[str, dict] = {}
for r in spark.sql(f"""
    SELECT table_name, table_type
    FROM `{_esc(cat)}`.information_schema.tables
    WHERE table_schema = '{_esc(sch)}' {view_filter}
    ORDER BY table_name
""").collect():
    if CFG["table_filter"].strip():
        import re as _re
        if not _re.search(CFG["table_filter"], r.table_name, _re.I):
            continue
    layer = _detect_layer(CFG["layer"], sch, r.table_name)
    tables_meta[r.table_name] = {
        "table_type": r.table_type,
        "layer":      layer,
        "cols":       [],   # filled below
    }

if not tables_meta:
    raise ValueError(f"No tables found in {cat}.{sch}")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables_meta)
cols_meta: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM `{_esc(cat)}`.information_schema.columns
    WHERE table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    cols_meta[r.table_name].append((r.column_name, r.data_type.upper()))
    tables_meta[r.table_name]["cols"].append((r.column_name, r.data_type.upper()))

total_cols = sum(len(v) for v in cols_meta.values())
print(f"Scope      : {cat}.{sch}")
print(f"Tables     : {len(tables_meta)}")
print(f"Columns    : {total_cols}")
print(f"Layer cfg  : {CFG['layer']}")
print()
layer_counts: Dict[str, int] = {}
for inv in tables_meta.values():
    layer_counts[inv["layer"]] = layer_counts.get(inv["layer"], 0) + 1
for lyr, cnt in sorted(layer_counts.items()):
    print(f"  {lyr}: {cnt} table(s)")


In [0]:
# ---------------------------------------------------------------------------
# SCHEMA ANALYZER -- applies all DETERMINISTIC naming checks.
# Reads information_schema only -- no data reads from actual tables.
# Collects findings_pre (raw), AI will refine NON_DESCRIPTIVE_NAME and
# TABLE_ENCODED_IN_KEY in the next cell.
#
# Per-table checks:
#   - Table name: snake_case, env tokens, view suffix, dim/fact prefix/suffix
#   - Column names: snake_case, reserved words, env tokens, opaque names,
#     dt/ts suffix, bool prefix/type, amount suffix, code/desc pairing,
#     source prefix, leading underscore, audit columns
# ---------------------------------------------------------------------------

findings_pre: List[dict] = []   # raw findings before AI pass

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _raw(tbl, col, tag, msg, layer="Unknown", dtype="", confidence="high",
         auto_action=None) -> dict:
    options = STANDARDIZATION_RULES.get(tag, [])
    decided = auto_action
    return {
        "table_name":    tbl,
        "column_name":   col,
        "classification":tag,
        "rule_ref":      TAGS.get(tag, "§?"),
        "layer":         layer,
        "data_type":     dtype,
        "message":       msg,
        "confidence":    confidence,
        "needs_steward_review": decided is None,
        "decided_action":       decided,
        "std_options":   options,
        "ai_reviewed":   False,
    }


def _check_table_name(tbl: str, layer: str) -> None:
    tbl_l = tbl.lower()

    # QUOTED_MIXED_CASE: UC stores mixed-case only when created with quoted identifier
    if RE_MIXED_CASE.search(tbl) and not RE_SNAKE_CASE.match(tbl):
        findings_pre.append(_raw(tbl, "--table--", "QUOTED_MIXED_CASE",
            f"Table '{tbl}' contains uppercase letters -- was created with a quoted identifier. "
            "UC stores names consistently when using lowercase unquoted identifiers (§1.8).",
            layer=layer))

    # NON_SNAKE_CASE (only if not caught by QUOTED_MIXED_CASE)
    if not RE_SNAKE_CASE.match(tbl_l) or RE_MIXED_CASE.search(tbl):
        if not RE_MIXED_CASE.search(tbl):  # already caught above
            findings_pre.append(_raw(tbl, "--table--", "NON_SNAKE_CASE",
                f"Table name '{tbl}' is not snake_case (§1.1). "
                "All identifiers must use snake_case.",
                layer=layer))

    # ENV_IN_NAME
    tokens = set(tbl_l.split("_"))
    env_found = tokens & ENV_TOKENS
    if env_found:
        findings_pre.append(_raw(tbl, "--table--", "ENV_IN_NAME",
            f"Table name '{tbl}' contains environment token(s): {env_found}. "
            "Express environment via catalog/workspace/bundle target (§1.5).",
            layer=layer))

    # VIEW_SUFFIX_PRESENT
    if RE_VIEW_SUFFIX.search(tbl_l) or RE_VIEW_PREFIX.match(tbl_l):
        findings_pre.append(_raw(tbl, "--table--", "VIEW_SUFFIX_PRESENT",
            f"Table/view name '{tbl}' contains a view-indicating suffix/prefix "
            "(_vw, _view, vw_). Prohibited in new models (§3a.4). "
            "Layer and UC object type convey shape; use materialized Delta tables.",
            layer=layer))

    # DIM_FACT_WRONG_LAYER: dim_/fact_ on non-Gold
    if layer in ("Bronze", "Silver"):
        if RE_DIM_PREFIX.match(tbl_l) or RE_FACT_PREFIX.match(tbl_l):
            findings_pre.append(_raw(tbl, "--table--", "DIM_FACT_WRONG_LAYER",
                f"Table '{tbl}' uses dim_/fact_ prefix in {layer} layer. "
                "These prefixes are for Gold star-schema only (§3a.2).",
                layer=layer))

    # DIM_FACT_SUFFIX_FORM
    if RE_DIM_SUFFIX.search(tbl_l) or RE_FACT_SUFFIX.search(tbl_l):
        findings_pre.append(_raw(tbl, "--table--", "DIM_FACT_SUFFIX_FORM",
            f"Table '{tbl}' uses suffix form (provider_dim / claims_fact). "
            "Use prefix form: dim_provider, fact_claims (§3a.5).",
            layer=layer))

    # DIM_FACT_ABBREVIATED
    if RE_DIM_ABBREV.match(tbl_l) or RE_FACT_ABBREV.match(tbl_l):
        prefix = tbl_l.split("_")[0] + "_"
        findings_pre.append(_raw(tbl, "--table--", "DIM_FACT_ABBREVIATED",
            f"Table '{tbl}' uses abbreviated prefix '{prefix}'. "
            "Full words required: dim_ and fact_ (§3a.1).",
            layer=layer))

    # STG_PREFIX_WRONG_LAYER: stg_ on Bronze/Silver is ambiguous
    if RE_STG_PREFIX.match(tbl_l) and layer in ("Bronze", "Silver"):
        findings_pre.append(_raw(tbl, "--table--", "STG_PREFIX_WRONG_LAYER",
            f"Table '{tbl}' uses stg_ prefix in {layer} layer. "
            "stg_ on Gold data_marts = analytical staging (§3a.7); "
            "on Bronze/Silver use medallion naming conventions instead.",
            layer=layer, confidence="medium"))


def _check_col_name(tbl: str, col: str, dtype: str, all_cols: Set[str],
                    layer: str) -> None:
    col_l = col.lower()

    # QUOTED_MIXED_CASE
    if RE_MIXED_CASE.search(col) and not RE_SNAKE_CASE.match(col):
        findings_pre.append(_raw(tbl, col, "QUOTED_MIXED_CASE",
            f"Column '{col}' contains uppercase -- was created with quoted identifier (§1.8).",
            layer=layer, dtype=dtype))
        return  # non-snake issues follow from this

    # NON_SNAKE_CASE
    if not RE_SNAKE_CASE.match(col_l):
        reason = ("camelCase" if RE_CAMEL_CASE.search(col) else
                  "PascalCase" if RE_PASCAL_CASE.match(col) else
                  "kebab-case" if RE_KEBAB_CASE.search(col) else
                  "UPPER_CASE" if RE_UPPER_CASE.match(col) else
                  "contains space" if RE_SPACE.search(col) else "non-standard")
        findings_pre.append(_raw(tbl, col, "NON_SNAKE_CASE",
            f"Column '{col}' is not snake_case ({reason}) (§1.1).",
            layer=layer, dtype=dtype))

    # RESERVED_WORD
    if col_l in SQL_RESERVED:
        findings_pre.append(_raw(tbl, col, "RESERVED_WORD",
            f"Column '{col}' is a SQL reserved word (§1.4). "
            f"Suffix or qualify: {col}_status, {col}_date, etc.",
            layer=layer, dtype=dtype))

    # ENV_IN_NAME
    tokens_c = set(col_l.split("_"))
    env_found = tokens_c & ENV_TOKENS
    if env_found:
        findings_pre.append(_raw(tbl, col, "ENV_IN_NAME",
            f"Column '{col}' contains environment token(s): {env_found} (§1.5).",
            layer=layer, dtype=dtype))

    # OPAQUE_COLUMN_NAME
    if RE_OPAQUE_NUMBERED.match(col_l) or RE_SINGLE_CHAR.match(col_l):
        findings_pre.append(_raw(tbl, col, "OPAQUE_COLUMN_NAME",
            f"Column '{col}' is an opaque name (§2f.1). "
            "Use governed descriptive names; apply UC phi_class tags for PHI.",
            layer=layer, dtype=dtype))

    # DT_TS_SUFFIX
    if RE_DT_SUFFIX.search(col_l):
        findings_pre.append(_raw(tbl, col, "DT_TS_SUFFIX",
            f"Column '{col}' uses _dt or _ts suffix. "
            "Use _date for dates, _at_est for Eastern timestamps, _at_utc for UTC (§2b.1).",
            layer=layer, dtype=dtype))

    # MISSING_EST_COMPANION: timestamp-like column without a paired _at_est
    if RE_AT_SUFFIX.search(col_l) and not RE_AT_EST.search(col_l):
        base = RE_AT_SUFFIX.sub("", col_l)
        est_col = base + "_at_est"
        if est_col not in all_cols:
            findings_pre.append(_raw(tbl, col, "MISSING_EST_COMPANION",
                f"Timestamp column '{col}' has no _at_est Eastern canonical companion "
                f"('{est_col}' not found in table). "
                "Business event timestamps must have _at_est canonical (§2b.2).",
                layer=layer, dtype=dtype, confidence="medium"))

    # DATE_WRONG_TYPE: column named _date but not typed DATE
    if RE_DATE_SUFFIX.search(col_l) and dtype not in ("DATE",) and dtype != "":
        if dtype in ("STRING", "VARCHAR", "CHAR", "TIMESTAMP", "TIMESTAMP_NTZ"):
            findings_pre.append(_raw(tbl, col, "DATE_WRONG_TYPE",
                f"Column '{col}' is named _date but has type {dtype} (not DATE) (§2b.3).",
                layer=layer, dtype=dtype))

    # BOOL_WRONG_PREFIX: flag/indicator-like column not prefixed is_/has_/was_
    if RE_BOOL_LIKE.search(col_l) and not RE_BOOL_PREFIX.match(col_l):
        findings_pre.append(_raw(tbl, col, "BOOL_WRONG_PREFIX",
            f"Column '{col}' appears to be a boolean/flag but lacks is_/has_/was_ prefix (§2c). "
            f"Rename to is_{col} or has_{col} as appropriate.",
            layer=layer, dtype=dtype))

    # BOOL_WRONG_TYPE: is_/has_/was_ column not typed correctly
    if RE_BOOL_PREFIX.match(col_l) and dtype not in BOOL_TYPES and dtype not in ("", "UNKNOWN"):
        findings_pre.append(_raw(tbl, col, "BOOL_WRONG_TYPE",
            f"Column '{col}' is prefixed is_/has_/was_ but has type {dtype} "
            f"(expected BOOLEAN or INT) (§2c.1).",
            layer=layer, dtype=dtype))

    # AMOUNT_NO_SUFFIX: monetary-looking column without _amount suffix
    if RE_AMOUNT_LIKE.search(col_l) and not RE_AMOUNT_SUFFIX.search(col_l):
        if dtype in NUMERIC_TYPES or dtype == "":
            findings_pre.append(_raw(tbl, col, "AMOUNT_NO_SUFFIX",
                f"Column '{col}' appears to be a monetary measure but lacks _amount suffix (§2d.1). "
                f"Rename to {col}_amount.",
                layer=layer, dtype=dtype))

    # SOURCE_PREFIX_IN_SILVER: source_ or EHR-specific prefix in Silver/Gold
    if RE_SOURCE_PREFIX.match(col_l) and layer in ("Silver", "Gold"):
        findings_pre.append(_raw(tbl, col, "SOURCE_PREFIX_IN_SILVER",
            f"Column '{col}' has a source-system prefix in {layer} layer (§2d.3). "
            "Bronze may retain source_ prefixes; Silver+ should use canonical names.",
            layer=layer, dtype=dtype))

    # LEADING_UNDERSCORE_MISUSE: leading _ not in approved audit set
    if col_l.startswith("_") and col_l not in APPROVED_LEADING_UNDERSCORE:
        findings_pre.append(_raw(tbl, col, "LEADING_UNDERSCORE_MISUSE",
            f"Column '{col}' has a leading underscore but is not an approved audit/metadata column (§2e.1). "
            f"Approved: {sorted(APPROVED_LEADING_UNDERSCORE)}.",
            layer=layer, dtype=dtype))

    # TABLE_ENCODED_IN_KEY: foreign key with table name baked in
    # Mark for AI review -- heuristic only here
    if col_l.endswith("_id") or col_l.endswith("_key"):
        parts = col_l.rsplit("_", 2)
        if len(parts) >= 3:
            # e.g. claims_table_member_id → middle part looks like 'table' or another table name
            middle = "_".join(parts[:-2])  # everything before the last _id/_key
            if any(kw in middle for kw in ("table", "tbl", "ref", "lookup")):
                findings_pre.append(_raw(tbl, col, "TABLE_ENCODED_IN_KEY",
                    f"Foreign key column '{col}' may encode a table name in the middle segment. "
                    "Foreign keys use the same name as the referenced key (§2a.2). "
                    "AI will review this finding.",
                    layer=layer, dtype=dtype, confidence="low"))


def _check_code_desc_pairing(tbl: str, all_col_lower: Set[str], layer: str) -> None:
    for col in all_col_lower:
        if RE_CODE_SUFFIX.search(col):
            domain = RE_CODE_SUFFIX.sub("", col)
            desc_col = domain + "_description"
            if desc_col not in all_col_lower and domain + "_desc" not in all_col_lower:
                findings_pre.append(_raw(tbl, col, "CODE_WITHOUT_DESCRIPTION",
                    f"Column '{col}' has no paired {domain}_description column (§2d.2). "
                    "Codes and descriptions must be paired consistently.",
                    layer=layer))
        if RE_DESC_SUFFIX.search(col):
            domain = RE_DESC_SUFFIX.sub("", col)
            if domain + "_code" not in all_col_lower:
                findings_pre.append(_raw(tbl, col, "DESCRIPTION_WITHOUT_CODE",
                    f"Column '{col}' has no paired {domain}_code column (§2d.2). "
                    "Codes and descriptions must be paired consistently.",
                    layer=layer))


def _check_audit_cols(tbl: str, all_col_lower: Set[str], layer: str) -> None:
    if layer != "Bronze":
        return
    missing = REQUIRED_AUDIT_BRONZE - all_col_lower
    if missing:
        findings_pre.append(_raw(tbl, "--table--", "AUDIT_COL_MISSING",
            f"Bronze table '{tbl}' is missing mandatory audit column(s): {sorted(missing)} (§2e.1). "
            "Audit columns are required on all Bronze tables.",
            layer=layer))


# ── Run deterministic checks ──────────────────────────────────────────────────
for tbl, inv in tables_meta.items():
    layer     = inv["layer"]
    cols      = inv["cols"]  # [(col_name, dtype), ...]
    col_lower = {c.lower() for c, _ in cols}

    if CFG["check_table_names"]:
        _check_table_name(tbl, layer)

    for col, dtype in cols:
        _check_col_name(tbl, col, dtype, col_lower, layer)

    _check_code_desc_pairing(tbl, col_lower, layer)
    _check_audit_cols(tbl, col_lower, layer)

print(f"Schema analysis done. {len(findings_pre)} raw finding(s) before AI pass.")
by_tag: Dict[str, int] = {}
for f in findings_pre:
    by_tag[f["classification"]] = by_tag.get(f["classification"], 0) + 1
for tag, cnt in sorted(by_tag.items(), key=lambda x: -x[1]):
    print(f"  {tag}: {cnt}")


In [0]:
# ---------------------------------------------------------------------------
# AI CLASSIFIER -- refines two ambiguous finding types:
# 1. NON_DESCRIPTIVE_NAME candidates: AI checks if name is a known
#    healthcare domain abbreviation (npi, icd, cpt, dob, etc.) vs.
#    genuinely opaque (mcl, tbl1, tmp_x).
# 2. TABLE_ENCODED_IN_KEY: AI confirms whether a key column genuinely
#    encodes a table name vs. a legitimate composite term.
#
# Also generates NON_DESCRIPTIVE_NAME findings for column names that
# passed regex checks but are still ambiguous abbreviations.
# ---------------------------------------------------------------------------

BATCH_SIZE = 20

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


# Step 1: Validate TABLE_ENCODED_IN_KEY candidates (low-confidence pre-findings)
tbl_key_candidates = [
    f for f in findings_pre
    if f["classification"] == "TABLE_ENCODED_IN_KEY" and f["confidence"] == "low"
]
confirmed_tbl_key: Set[Tuple[str,str]] = set()

if CFG["enable_ai"] and tbl_key_candidates:
    for chunk in _chunked(tbl_key_candidates):
        payload = json.dumps([{
            "table":  f["table_name"],
            "column": f["column_name"],
            "all_table_cols": [c for c, _ in tables_meta.get(f["table_name"], {}).get("cols", [])][:30],
        } for f in chunk], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            "For each column, determine if its name GENUINELY encodes a referenced table name "
            "(a violation per MOSAIC §2a.2). "
            "Example violation: 'claims_table_member_id' -- the word 'table' or table name 'claims' "
            "is embedded in the foreign key name. "
            "Not a violation: 'member_claim_line_id' -- this is a legitimate composite term. "
            f"Items: {payload}. "
            'Return: [{"table":"<t>","column":"<c>","is_violation":true|false,'
            '"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                if item.get("is_violation") and item.get("confidence") in ("high","medium"):
                    confirmed_tbl_key.add((item.get("table",""), item.get("column","")))
        except Exception as e:
            print(f"  [WARN] AI TABLE_ENCODED_IN_KEY pass failed: {e}")
            # Conservative: confirm all low-confidence ones
            for f in chunk:
                confirmed_tbl_key.add((f["table_name"], f["column_name"]))

    # Upgrade confirmed findings to high confidence; remove unconfirmed
    findings_pre[:] = [
        f if not (f["classification"] == "TABLE_ENCODED_IN_KEY" and f["confidence"] == "low")
        else {**f, "confidence": "high"} if (f["table_name"], f["column_name"]) in confirmed_tbl_key
        else None
        for f in findings_pre
    ]
    findings_pre[:] = [f for f in findings_pre if f is not None]

# Step 2: NON_DESCRIPTIVE_NAME check via AI
# Gather short column names (2-5 chars without underscores) and opaque abbreviations
# that passed regex but might still be non-descriptive
ai_name_candidates: List[dict] = []
for tbl, inv in tables_meta.items():
    layer = inv["layer"]
    for col, dtype in inv["cols"]:
        col_l = col.lower()
        # Skip already flagged columns
        already_flagged = any(
            f["table_name"] == tbl and f["column_name"] == col
            for f in findings_pre
            if f["classification"] in ("NON_SNAKE_CASE","QUOTED_MIXED_CASE","OPAQUE_COLUMN_NAME")
        )
        if already_flagged:
            continue
        # Flag short non-standard abbreviations or tbl-prefixed names for AI review
        parts = col_l.split("_")
        is_short_abbrev = (len(col_l) <= 5 and "_" not in col_l and col_l not in SQL_RESERVED)
        has_tbl_prefix  = parts[0] in ("tbl", "t", "tmp", "temp", "col", "fld")
        if is_short_abbrev or has_tbl_prefix:
            ai_name_candidates.append({
                "table": tbl, "column": col, "dtype": dtype, "layer": layer
            })

if CFG["enable_ai"] and ai_name_candidates:
    for chunk in _chunked(ai_name_candidates):
        payload = json.dumps([{
            "table":  c["table"],
            "column": c["column"],
            "dtype":  c["dtype"],
        } for c in chunk], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            "For each column name, determine if it is a KNOWN and LEGITIMATE abbreviation in healthcare "
            "data engineering (e.g. npi, icd, cpt, dob, dod, ssn, dme, hmo, ppo, aco, ehr, emr, "
            "edi, cms, fhir, loinc, snomed, rxnorm, upin, tin, ein, apc, drg, rev) "
            "OR an opaque/non-descriptive abbreviation that violates MOSAIC §1.2. "
            "Also flag any name using tbl_, t_, tmp_, temp_, col_, fld_ prefixes. "
            f"Items: {payload}. "
            'Return: [{"table":"<t>","column":"<c>","is_non_descriptive":true|false,'
            '"reason":"<why>","confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                if item.get("is_non_descriptive") and item.get("confidence") in ("high","medium"):
                    findings_pre.append({
                        "table_name":    item.get("table",""),
                        "column_name":   item.get("column",""),
                        "classification":"NON_DESCRIPTIVE_NAME",
                        "rule_ref":      TAGS["NON_DESCRIPTIVE_NAME"],
                        "layer":         next((c["layer"] for c in chunk if c["column"] == item.get("column","")), "Unknown"),
                        "data_type":     next((c["dtype"] for c in chunk if c["column"] == item.get("column","")), ""),
                        "message":       f"Column '{item.get('column','')}' is a non-descriptive name (§1.2). "
                                         f"{item.get('reason','')} Rename or register in approved abbreviation glossary.",
                        "confidence":    item.get("confidence","medium"),
                        "needs_steward_review": True,
                        "decided_action": None,
                        "std_options":   STANDARDIZATION_RULES.get("NON_DESCRIPTIVE_NAME",[]),
                        "ai_reviewed":   True,
                    })
        except Exception as e:
            print(f"  [WARN] AI NON_DESCRIPTIVE_NAME pass failed: {e}")

print(f"AI pass done. {len(findings_pre)} total finding(s).")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE -- converts raw findings_pre into structured findings_out.
# Deduplicates (same table+col+tag), adds severity, formats for Delta write.
# ---------------------------------------------------------------------------

def _finding(raw: dict) -> dict:
    options = raw.get("std_options", STANDARDIZATION_RULES.get(raw["classification"], []))
    decided = raw.get("decided_action")
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               raw["table_name"],
        "column_name":              raw["column_name"],
        "layer":                    raw.get("layer", "Unknown"),
        "data_type":                raw.get("data_type", ""),
        "rule_ref":                 raw.get("rule_ref", TAGS.get(raw["classification"], "§?")),
        "classification":           raw["classification"],
        "severity":                 SEVERITY.get(raw["classification"], 5),
        "message":                  raw["message"],
        "sample_values":            json.dumps([], default=str),
        "recommended_action":       (raw.get("std_options") or
                                     STANDARDIZATION_RULES.get(raw["classification"]) or [{}]
                                    )[0].get("label", ""),
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               raw.get("confidence", "high"),
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


# Deduplicate: same (table, col, classification) → keep highest confidence
seen: Dict[Tuple[str,str,str], dict] = {}
CONFIDENCE_RANK = {"high": 3, "medium": 2, "low": 1}
for raw in findings_pre:
    key = (raw["table_name"], raw["column_name"], raw["classification"])
    prev = seen.get(key)
    if prev is None or CONFIDENCE_RANK.get(raw.get("confidence","low"),0) > CONFIDENCE_RANK.get(prev.get("confidence","low"),0):
        seen[key] = raw

all_findings: List[dict] = [_finding(raw) for raw in seen.values()]

# Sort by severity desc, then table, then column
all_findings.sort(key=lambda f: (-f["severity"], f["table_name"], f["column_name"]))

print(f"Rule engine done. {len(all_findings)} finding(s) after deduplication.")
by_tag: Dict[str, int] = {}
for f in all_findings:
    by_tag[f["classification"]] = by_tag.get(f["classification"], 0) + 1
for tag, cnt in sorted(by_tag.items(), key=lambda x: -x[1])[:15]:
    print(f"  {tag}: {cnt}")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, IntegerType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",               StringType(),  True),
    StructField("run_ts",               StringType(),  True),
    StructField("catalog",              StringType(),  True),
    StructField("schema",               StringType(),  True),
    StructField("table_name",           StringType(),  True),
    StructField("column_name",          StringType(),  True),
    StructField("layer",                StringType(),  True),
    StructField("data_type",            StringType(),  True),
    StructField("rule_ref",             StringType(),  True),
    StructField("classification",       StringType(),  True),
    StructField("severity",             IntegerType(), True),
    StructField("message",              StringType(),  True),
    StructField("sample_values",        StringType(),  True),
    StructField("recommended_action",   StringType(),  True),
    StructField("standardization_rule", StringType(),  True),
    StructField("confidence",           StringType(),  True),
    StructField("needs_steward_review", BooleanType(), True),
    StructField("decided_action",       StringType(),  True),
    StructField("decided_by",           StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No naming findings. All objects appear compliant.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "RESERVED_WORD", "NON_SNAKE_CASE", "QUOTED_MIXED_CASE",
    "DIM_FACT_WRONG_LAYER", "VIEW_SUFFIX_PRESENT",
}

if not all_findings:
    print("No naming convention findings. All objects appear compliant.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("severity").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING / HIGH SEVERITY FINDINGS: {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","layer","data_type",
            "classification","rule_ref","severity","message",
            "recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By rule
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by rule")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","layer")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.max("severity").alias("max_severity"),
           ).orderBy(F.col("max_severity").desc(), F.col("findings").desc())
    )

    # Section 3 -- §1 Universal
    s1_df = fdf.filter(F.col("rule_ref") == "§1-Universal")
    n_s1  = s1_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 3 -- §1 Universal identifier violations ({n_s1})")
    print("  snake_case, reserved words, environment tokens, quoted mixed-case")
    print("-" * 70)
    if n_s1:
        display(s1_df.select(
            "table_name","column_name","layer","classification","message",
            "recommended_action","decided_action","decided_by"
        ).orderBy(F.col("severity").desc(),"table_name","column_name"))

    # Section 4 -- §2 Column conventions
    s2_df = fdf.filter(F.col("rule_ref").isin(
        "§2b-Dates","§2c-Booleans","§2d-Measures","§2e-Audit","§2f-Sensitive","§2a-Keys"
    ))
    n_s2  = s2_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- §2 Column naming convention violations ({n_s2})")
    print("-" * 70)
    if n_s2:
        display(s2_df.select(
            "table_name","column_name","layer","data_type","rule_ref","classification",
            "message","recommended_action","decided_action","decided_by"
        ).orderBy("rule_ref","table_name","column_name"))

    # Section 5 -- §3 Table naming
    s3_df = fdf.filter(F.col("rule_ref") == "§3-Table-naming")
    n_s3  = s3_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- §3 Table/view naming violations ({n_s3})")
    print("  dim_/fact_ layer compliance, view suffixes, abbreviated prefixes")
    print("-" * 70)
    if n_s3:
        display(s3_df.select(
            "table_name","column_name","layer","classification","message",
            "recommended_action","decided_action","decided_by"
        ).orderBy(F.col("severity").desc(),"table_name"))

    # Section 6 -- By table
    print("\n" + "-" * 70)
    print("SECTION 6 -- Findings by table")
    print("-" * 70)
    display(
        fdf.groupBy("table_name","layer")
           .agg(
               F.count("*").alias("total_findings"),
               F.sum(F.when(F.col("classification").isin(BLOCKING), 1).otherwise(0)).alias("blocking"),
               F.collect_set("classification").alias("violations"),
           ).orderBy(F.col("blocking").desc(), F.col("total_findings").desc())
    )

    # Section 7 -- Work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 7 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","layer","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Tables: {len(tables_meta)}  |  Columns: {total_cols}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  §1: {n_s1}  |  §2: {n_s2}  |  §3: {n_s3}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Renames of certified Gold columns require steward approval and consumer impact assessment (§1.7).")
